# Financial Market Statistical Analysis

This notebook provides a comprehensive statistical analysis of financial features using the refactored `finance_ml.analytics` module. 
It leverages advanced statistical methods (Bayesian, MCMC, Kalman Filters), interactive dashboards, and performance-optimized operations.


## Table of Contents

1. **Setup & Environment Configuration**
2. **Data Acquisition** — Load from `mv_all_stock_features`, backfill, validate
3. **Statistical Analysis by Category** (3.1–3.16) — Bayesian, distributions, visualizations
4. **Advanced Modeling** — Hierarchical MCMC, Gaussian copulas
5. **Enhanced Visualizations** — Valuation, Earnings Quality, Quality & Risk, Growth
6. **Stock Screening** — Quality, GARP, Value, Growth, Reversion, Integrity
7. **Summary Dashboard & Export**
8. **Analysis Summary**


## 1. Setup and Environment Configuration
We import the core analytics modules and configure the visualization environment.


In [1]:
import logging
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

# Configure database connection BEFORE importing analytics modules
if "DB_URL" not in os.environ:
    env_file = "environment_variables.txt"
    if os.path.exists(env_file):
        with open(env_file) as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    key, value = line.split("=", 1)
                    os.environ[key.strip()] = value.strip()
    else:
        raise ValueError("DB_URL environment variable not set and environment_variables.txt not found")
else:
    logging.info("DB_URL environment variable already set, skipping loading from file")

# ── Core analytics ──────────────────────────────────────────────────
from probabilistic_ml_model import (
    # Data utilities
    backfill_feature_columns,
    FeatureViewCatalog,
    compare_registry_with_local,
    _get_fallback_feature_categories,
    # Statistical analysis
    bayesian_category_analysis,
    fit_distributions_by_category,
    hierarchical_mcmc_by_sector,
    kalman_momentum_filter,
    fit_gaussian_copula,
    analyze_employee_productivity_frontier,
    detect_accounting_anomalies,
    analyze_reporting_lag_sentiment,
    # Optimized operations
    fast_ruin_probability,
    get_optimization_status,
    # Screening
    create_enhanced_screener,
    screen_garp_opportunities,
    screen_high_yield_safe_dividends,
    screen_value_opportunities,
    screen_growth_momentum,
    screen_valuation_reversion_candidates,
    screen_integrity_filtered_growth,
    # Feature analytics & dashboards
    PLOTLY_TEMPLATE,
    create_interactive_momentum_dashboard,
    create_interactive_valuation_heatmap,
    create_leverage_liquidity_quadrant,
    bayesian_earnings_beat_model,
    analyze_distress_distribution,
    create_summary_dashboard,
)

# ── Probability analytics ──────────────────────────────────────────
from probabilistic_ml_model import (
    EarningsBeatProbabilityModel,
    CreditRiskProbabilityModel,
    DividendCutProbabilityModel,
    PriceTargetAchievementModel,
    EPSStreakAnalyzer,
    ModelConfidenceEstimator,
    export_probability_analytics_results,
)

# ── Visualizations (all sub-modules) ──────────────────────────────
from probabilistic_ml_model.visualizations.earnings_quality import (
    create_earnings_surprise_dashboard,
    create_eps_trajectory_analysis,
    create_earnings_quality_decomposition,
    create_beat_rate_heatmap,
    create_earnings_consistency_matrix,
    create_revision_momentum_chart,
    create_gaap_divergence_plot,
    create_enhanced_beat_probability_dashboard,
)
from probabilistic_ml_model import (
    create_analyst_sentiment_histogram,
    create_analyst_upside_scatter,
    create_eps_surprise_histogram,
    create_eps_trajectory_scatter,
    create_growth_correlation_heatmap,
    create_revenue_vs_eps_growth_scatter,
    create_fcf_margin_yield_scatter,
    create_cash_flow_quality_boxplot,
    create_dividend_yield_payout_scatter,
    create_shareholder_yield_histogram,
    create_rnd_intensity_boxplot,
    create_rnd_intensity_growth_scatter,
    create_rnd_per_employee_histogram,
    create_inventory_days_turnover_scatter,
    create_goodwill_concentration_boxplot,
    create_goodwill_impairment_scatter,
    create_acquisition_activity_histogram,
    create_capex_growth_scatter,
    create_investment_efficiency_boxplot,
    create_ma_intensity_histogram,
    create_valuation_violin_plot,
    create_quality_risk_radar_chart,
    create_leverage_liquidity_bubble_chart,
    create_productivity_quadrant,
    create_accounting_quality_breakdown,
    create_valuation_range_visual,
)
from probabilistic_ml_model import (
    create_margin_waterfall_chart,
    create_dupont_decomposition_dashboard,
    create_profitability_quadrant,
    create_margin_trend_heatmap,
)
from probabilistic_ml_model import (
    create_momentum_ribbon_chart,
    create_52w_range_distribution,
    create_trend_strength_matrix,
    create_momentum_divergence_scatter,
)
from probabilistic_ml_model import (
    create_earnings_calendar_heatmap,
    create_inventory_cycle_analysis,
    create_fcf_trajectory_chart,
    create_dividend_streak_timeline,
)
from probabilistic_ml_model import (
    create_valuation_multiples_comparison,
    create_valuation_distribution_dashboard,
    create_relative_valuation_matrix,
    create_valuation_vs_growth_quadrant,
    create_historical_valuation_percentile,
)
from probabilistic_ml_model.visualizations.quality_risk import (
    create_piotroski_fscore_breakdown,
    create_altman_zscore_distribution,
    create_quality_risk_quadrant,
    create_beneish_mscore_analysis,
    create_risk_tier_sunburst,
    create_distress_early_warning_dashboard,
)
from probabilistic_ml_model.visualizations.growth_analysis import (
    create_growth_waterfall_chart,
    create_growth_consistency_matrix,
    create_growth_vs_profitability_quadrant,
    create_growth_acceleration_chart,
    create_sustainable_growth_analysis,
)

# Configuration
logging.basicConfig(level=logging.INFO)
warnings.filterwarnings("ignore")
px.defaults.template = PLOTLY_TEMPLATE

opt_status = get_optimization_status()
print(f"JIT Acceleration: {opt_status.get('numba_available')}")
print(f"Feature Categories Loaded: {len(FeatureViewCatalog)}")

# --- InferenceData schema (ArviZ / xarray bridge) ---
from probabilistic_ml_model.data_utils.inference_schema import (
    ARVIZ_AVAILABLE,
    build_category_analysis_inference_data,
    summarize_inference_data,
)

# --- Probabilistic visualizations (ArviZ-backed) ---
from probabilistic_ml_model.visualizations.probability_viz import (
    create_beat_probability_posterior,
    create_bayesian_category_ridge,
)


ImportError: cannot import name 'backfill_feature_columns' from 'probabilistic_ml_model' (C:\Users\markm\PycharmProjects\PML_Finance_Project\probabilistic_ml_model\__init__.py)

## 2. Data Acquisition
Loading feature categories dynamically from the `calculated_features_registry` table (with fallback).
Data is loaded from the `mv_all_stock_features` materialized view.


In [ ]:
%%sql
select * from public.calculated_features_registry;

In [ ]:
%%sql
SELECT *
FROM public.mv_all_stock_features
WHERE next_earnings >= current_date - INTERVAL '2 months' AND next_earnings <= current_date + INTERVAL '1 months'
ORDER BY next_earnings ASC;



In [ ]:
# Normalize SQL result and backfill expected columns
if not isinstance(df, pd.DataFrame):
    try:
        df = df.DataFrame()
    except AttributeError:
        df = pd.DataFrame(df)

if isinstance(df, pd.DataFrame) and len(df) > 0:
    df = backfill_feature_columns(df)
    print(f"✓ Loaded {len(df):,} stocks with {len(df.columns)} features after backfill")

    # Quick data quality check
    null_pct = df.isnull().mean().sort_values(ascending=False)
    high_null = null_pct[null_pct > 0.5]
    if len(high_null) > 0:
        print(f"⚠️ {len(high_null)} features with >50% null values")
else:
    raise ValueError("No data loaded from mv_all_stock_features — check DB connection and query")


## 3. Comprehensive Statistical Analysis by Category

This section provides in-depth statistical analysis across all 14 feature categories using Bayesian methods, distribution fitting, and specialized visualizations.


### 3.1 Valuation Ratios
We use Bayesian analysis to estimate true valuation means and visualize valuation metrics across industries.


In [ ]:
val_results = bayesian_category_analysis(df, 'Valuation Ratios', FeatureViewCatalog['Valuation Ratios'])
val_distributions = fit_distributions_by_category(df, 'Valuation Ratios', FeatureViewCatalog['Valuation Ratios'])
create_interactive_valuation_heatmap(df).show()


### 3.2 Technical Analysis
Applying Kalman filters to smooth momentum signals and visualizing multi-period momentum patterns.


In [ ]:
df_kalman = kalman_momentum_filter(df, momentum_cols=['price_momentum_1y', 'price_momentum_3m'])
momentum_results = bayesian_category_analysis(df, 'Technical Analysis', FeatureViewCatalog['Technical Analysis'])
create_interactive_momentum_dashboard(df).show()

In [ ]:
create_52w_range_distribution(df).show()

In [ ]:
create_momentum_ribbon_chart(df).show()


### 3.2a Trend Strength Matrix
Heatmap: long_term_trend_score vs secular_trend_flag by industry.


In [ ]:
create_trend_strength_matrix(df).show()


In [ ]:
# Momentum divergence: short-term vs long-term momentum
create_momentum_divergence_scatter(df).show()


### 3.3 Profitability
Utilizing DuPont decomposition and margin waterfall charts to analyze bottom-line drivers with Bayesian estimation.


In [ ]:
prof_results = bayesian_category_analysis(df, 'Profitability', FeatureViewCatalog['Profitability'])
prof_distributions = fit_distributions_by_category(df, 'Profitability', FeatureViewCatalog['Profitability'])
create_dupont_decomposition_dashboard(df).show()

In [ ]:
create_margin_waterfall_chart(df).show()

In [ ]:
create_profitability_quadrant(df).show()


### 3.3a Margin Trend Heatmap
Net margin trend analysis by industry.


In [ ]:
create_margin_trend_heatmap(df).show()


### 3.4 Quality & Risk
Assessing quality scores and distress risk using Bayesian analysis and tail risk metrics.


In [ ]:
quality_results = bayesian_category_analysis(df, 'Quality & Risk', FeatureViewCatalog['Quality & Risk'])
quality_distributions = fit_distributions_by_category(df, 'Quality & Risk', FeatureViewCatalog['Quality & Risk'])
analyze_distress_distribution(df).show()
df_ruin = fast_ruin_probability(df)


### 3.5 Leverage & Liquidity
Analyzing solvency metrics and balance sheet strength using quadrant analysis and Bayesian estimation.


In [ ]:
leverage_results = bayesian_category_analysis(df, 'Leverage & Liquidity', FeatureViewCatalog['Leverage & Liquidity'])
leverage_distributions = fit_distributions_by_category(df, 'Leverage & Liquidity',
                                                       FeatureViewCatalog['Leverage & Liquidity'])
create_leverage_liquidity_quadrant(df).show()


### 3.6 Analyst Sentiment
Analyzing analyst recommendations, price target upside, and EPS revision momentum.


In [ ]:
sentiment_results = bayesian_category_analysis(df, 'Analyst Sentiment', FeatureViewCatalog['Analyst Sentiment'])
sentiment_distributions = fit_distributions_by_category(df, 'Analyst Sentiment',
                                                        FeatureViewCatalog['Analyst Sentiment'])
create_analyst_sentiment_histogram(df).show()

In [ ]:
create_analyst_upside_scatter(df).show()


### 3.7 Earnings Quality
Evaluating earnings surprises, GAAP adjustments, and earnings trajectory with Bayesian methods.


In [ ]:
earnings_quality_results = bayesian_category_analysis(df, 'Earnings Quality', FeatureViewCatalog['Earnings Quality'])
earnings_quality_distributions = fit_distributions_by_category(df, 'Earnings Quality',
                                                               FeatureViewCatalog['Earnings Quality'])
earnings_beat_probs = bayesian_earnings_beat_model(df)
create_eps_surprise_histogram(df).show()

In [ ]:
create_eps_trajectory_scatter(df).show()


### 3.8 Growth Metrics
Analyzing revenue, EBITDA, EPS, and FCF growth patterns with distribution fitting.


In [ ]:
growth_results = bayesian_category_analysis(df, 'Growth Metrics', FeatureViewCatalog['Growth Metrics'])
growth_distributions = fit_distributions_by_category(df, 'Growth Metrics', FeatureViewCatalog['Growth Metrics'])
create_growth_correlation_heatmap(df, FeatureViewCatalog['Growth Metrics']).show()

In [ ]:
create_revenue_vs_eps_growth_scatter(df).show()


### 3.9 Cash Flow
Analyzing free cash flow metrics, self-funding ratios, and cash flow quality.


In [ ]:
cashflow_results = bayesian_category_analysis(df, 'Cash Flow', FeatureViewCatalog['Cash Flow'])
cashflow_distributions = fit_distributions_by_category(df, 'Cash Flow', FeatureViewCatalog['Cash Flow'])
create_fcf_trajectory_chart(df).show()

In [ ]:
create_fcf_margin_yield_scatter(df).show()

In [ ]:
create_cash_flow_quality_boxplot(df).show()


### 3.10 Dividend Reliability
Evaluating dividend sustainability, payout ratios, and shareholder yield.


In [ ]:
dividend_cat = 'Dividend Features' if 'Dividend Features' in FeatureViewCatalog else 'Dividend Reliability'
dividend_results = bayesian_category_analysis(df, dividend_cat, FeatureViewCatalog.get(dividend_cat, []))
dividend_distributions = fit_distributions_by_category(df, dividend_cat, FeatureViewCatalog.get(dividend_cat, []))
create_dividend_streak_timeline(df).show()

In [ ]:
create_dividend_yield_payout_scatter(df).show()

In [ ]:
create_shareholder_yield_histogram(df).show()


### 3.11 R&D Investment
Analyzing R&D intensity, growth patterns, and innovation investment efficiency.


In [ ]:
rnd_cat = next((c for c in ['R&D Investment', 'Efficiency Ratios'] if c in FeatureViewCatalog), None)
if rnd_cat:
    rnd_results = bayesian_category_analysis(df, rnd_cat, FeatureViewCatalog[rnd_cat])
    rnd_distributions = fit_distributions_by_category(df, rnd_cat, FeatureViewCatalog[rnd_cat])
create_rnd_intensity_boxplot(df).show()

In [ ]:
create_rnd_intensity_growth_scatter(df).show()

In [ ]:
create_rnd_per_employee_histogram(df).show()


### 3.12 Inventory Temporal
Analyzing inventory cycles, turnover efficiency, and buildup patterns.


In [ ]:
inventory_cat = next((c for c in ['Inventory Temporal', 'Balance Sheet'] if c in FeatureViewCatalog), None)
if inventory_cat:
    inventory_results = bayesian_category_analysis(df, inventory_cat, FeatureViewCatalog[inventory_cat])
    inventory_distributions = fit_distributions_by_category(df, inventory_cat, FeatureViewCatalog[inventory_cat])
create_inventory_cycle_analysis(df).show()

In [ ]:
create_inventory_days_turnover_scatter(df).show()


### 3.13 Goodwill & M&A
Evaluating acquisition activity, goodwill concentration, and impairment risk.


In [ ]:
goodwill_cat = next((c for c in ['Goodwill & M&A', 'Accounting Quality'] if c in FeatureViewCatalog), None)
if goodwill_cat:
    goodwill_results = bayesian_category_analysis(df, goodwill_cat, FeatureViewCatalog[goodwill_cat])
    goodwill_distributions = fit_distributions_by_category(df, goodwill_cat, FeatureViewCatalog[goodwill_cat])
create_goodwill_concentration_boxplot(df).show()

In [ ]:
create_goodwill_impairment_scatter(df).show()

In [ ]:
create_acquisition_activity_histogram(df).show()


### 3.14 CapEx & Investment
Analyzing capital expenditure patterns, investment efficiency, and M&A intensity.


In [ ]:
capex_results = bayesian_category_analysis(df, 'Efficiency Ratios', FeatureViewCatalog['Efficiency Ratios'])
capex_distributions = fit_distributions_by_category(df, 'Efficiency Ratios', FeatureViewCatalog['Efficiency Ratios'])
create_capex_growth_scatter(df).show()

In [ ]:
create_investment_efficiency_boxplot(df).show()

In [ ]:
create_ma_intensity_histogram(df).show()


### 3.15 Employee Productivity & Accounting Integrity
Analyzing labor efficiency and identifying accounting anomalies using distribution fitting.


In [ ]:
# Analyze employee productivity
df = analyze_employee_productivity_frontier(df)
create_productivity_quadrant(df).show()


In [ ]:
# Detect accounting anomalies
df = detect_accounting_anomalies(df)
if len(df) > 0:
    ticker = df.iloc[0]['ticker']
    create_accounting_quality_breakdown(df, ticker).show()


In [ ]:
# Reporting lag sentiment analysis
lag_results = analyze_reporting_lag_sentiment(df)
if lag_results:
    print(f"📊 Reporting Lag Sentiment Correlation: {lag_results.get('correlation', 0):.4f}")
    print(f"   P-value: {lag_results.get('p_value', 0):.4f}")


### 3.16 Probability Analytics: Earnings Beat & EPS Streaks
Advanced probability analysis using Bayesian Beta-Binomial models for earnings beat prediction,
Markov chain-style EPS streak analysis, and model confidence estimation with calibration metrics.


In [ ]:
# Initialize probability analytics models
beat_model = EarningsBeatProbabilityModel()
streak_analyzer = EPSStreakAnalyzer(mean_reversion_weight=0.3)
confidence_estimator = ModelConfidenceEstimator(n_bins=10)
credit_model = CreditRiskProbabilityModel()
dividend_model = DividendCutProbabilityModel()
pt_model = PriceTargetAchievementModel()

# Compute enhanced Bayesian earnings beat probabilities using three-layer fusion
# (historical beats, revision momentum, GAAP quality guard)
# Uses dynamic total_reports derived from non-null reported EPS data
probability_results = beat_model.analyze_dataframe_enhanced(
    df,
    sector_col='sector' if 'sector' in df.columns else 'industry',
    ticker_col='ticker'
)

# Run new probability models
credit_results = credit_model.analyze_dataframe(df)
dividend_results = dividend_model.analyze_dataframe(df)
pt_results = pt_model.analyze_dataframe(df)

print(f"📊 Probability Analytics Summary")
print(f"   Stocks analyzed (Earnings Beat): {len(probability_results)}")
print(f"   Stocks analyzed (Credit Risk): {len(credit_results)}")
print(f"   Stocks analyzed (Dividend Cut): {len(dividend_results)}")
print(f"   Stocks analyzed (Price Target): {len(pt_results)}")

if len(probability_results) > 0:
    likely_beat = (probability_results['beat_classification'] == 'likely_beat').sum()
    print(f"   Classified as 'likely beat': {likely_beat} ({likely_beat / len(probability_results) * 100:.1f}%)")
    print(f"   Mean posterior beat probability: {probability_results['posterior_beat_prob'].mean():.1%}")
    if 'dynamic_total_reports' in probability_results.columns:
        avg_dynamic = probability_results['dynamic_total_reports'].mean()
        print(f"   Avg dynamic total reports per stock: {avg_dynamic:.1f}")
    if 'data_source' in probability_results.columns:
        source_counts = probability_results['data_source'].value_counts()
        for src, cnt in source_counts.items():
            print(f"   Data source '{src}': {cnt} stocks")


In [ ]:
# Display top stocks by Credit Risk probability
if len(credit_results) > 0:
    print("\n⚠️ Top 50 Stocks by Credit Risk (Financial Distress Probability):")
    display(credit_results.nlargest(50, 'distress_probability')[
                ['ticker', 'name', 'distress_probability', 'risk_level', 'altman_z_score', 'cash_runway_months']
            ])


In [ ]:
# Display top stocks by Dividend Cut probability
if len(dividend_results) > 0:
    print("\n💰 Top 50 Stocks by Dividend Cut Probability:")
    display(dividend_results.nlargest(50, 'dividend_cut_probability')[
                ['ticker', 'name', 'dividend_cut_probability', 'risk_category', 'fcf_dividend_coverage', 'payout_ratio']
            ])


In [ ]:
# Display top stocks by Price Target Achievement probability
if len(pt_results) > 0:
    print("\n🎯 Top 50 Stocks by Price Target Achievement Probability:")
    display(pt_results.nlargest(50, 'achievement_probability')[
                ['ticker', 'name', 'achievement_probability', 'upside_potential', 'analyst_rating_normalized']
            ])


In [ ]:
# Display top stocks by posterior beat probability
if len(probability_results) > 0:
    print("\n🎯 Top 50 Stocks by Posterior Beat Probability:")
    display_cols = ['ticker', 'name', 'historical_beat_rate', 'posterior_beat_prob',
                    'ci_90_lower', 'ci_90_upper', 'confidence_score', 'beat_classification']
    # Include new dynamic and forward estimate columns when available
    for extra in ['dynamic_total_reports', 'revision_momentum_score', 'eps_norm_est_fy1e',
                  'next_earnings_status', 'data_source']:
        if extra in probability_results.columns:
            display_cols.append(extra)
    top_beat_prob = probability_results.nlargest(50, 'posterior_beat_prob')[
        [c for c in display_cols if c in probability_results.columns]
    ]
    display(top_beat_prob)


In [ ]:
# Enhanced Earnings Beat Probability Dashboard (three-layer fusion)
create_enhanced_beat_probability_dashboard(probability_results).show()


In [ ]:
# Revision Momentum Chart
if 'revision_momentum_score' in probability_results.columns:
    create_revision_momentum_chart(probability_results).show()


In [ ]:
# GAAP Divergence Plot
if 'gaap_norm_spread' in probability_results.columns:
    create_gaap_divergence_plot(probability_results).show()


#### EPS Streak Analysis
Analyzing earnings beat/miss streaks with continuation and mean reversion probabilities.


In [ ]:
# Compute EPS streak analysis with dynamic totals and forward estimate integration
streak_results = streak_analyzer.analyze_dataframe(
    df,
    trajectory_col='eps_trajectory_score',
    streak_col='eps_positive_streak' if 'eps_positive_streak' in df.columns else None,
    ticker_col='ticker'
)

print(f"📈 EPS Streak Analysis")
print(f"   Stocks analyzed: {len(streak_results)}")
if len(streak_results) > 0:
    beat_streaks = (streak_results['streak_type'] == 'beat').sum()
    miss_streaks = (streak_results['streak_type'] == 'miss').sum()
    print(f"   On beat streaks: {beat_streaks}")
    print(f"   On miss streaks: {miss_streaks}")
    print(f"   Mean continuation probability: {streak_results['continuation_probability'].mean():.1%}")
    print(f"   Mean reversion probability: {streak_results['mean_reversion_probability'].mean():.1%}")
    if 'dynamic_total_reports' in streak_results.columns:
        avg_total = streak_results['dynamic_total_reports'].mean()
        print(f"   Avg dynamic total reports per stock: {avg_total:.1f}")
    if 'historical_beat_rate' in streak_results.columns:
        print(f"   Mean historical beat rate: {streak_results['historical_beat_rate'].mean():.1%}")


In [ ]:
# Display stocks with strongest beat streaks (including dynamic totals and forward signals)
if len(streak_results) > 0:
    display_cols = ['ticker', 'name', 'current_streak', 'streak_type', 'continuation_probability',
                    'mean_reversion_probability', 'expected_next_outcome', 'prediction_confidence']
    for extra in ['dynamic_total_reports', 'historical_beat_rate', 'revision_momentum_score',
                  'next_earnings_status']:
        if extra in streak_results.columns:
            display_cols.append(extra)
    strong_streaks = streak_results[streak_results['streak_type'] == 'beat'].nlargest(50, 'current_streak')[
        [c for c in display_cols if c in streak_results.columns]
    ]
    display(strong_streaks)


#### Model Confidence & Calibration
Assessing model reliability using Brier score, calibration error, and AUC-ROC metrics.


In [ ]:
# Compute model confidence metrics (using simulated outcomes for demonstration)
if len(probability_results) > 10:
    np.random.seed(42)
    # Simulate actual outcomes based on posterior probability
    simulated_outcomes = (
            np.random.random(len(probability_results))
            < probability_results['posterior_beat_prob'].values
    ).astype(float)

    confidence_result = confidence_estimator.compute_confidence_metrics(
        predicted_probs=probability_results['posterior_beat_prob'].values,
        actual_outcomes=simulated_outcomes,
        model_name='Bayesian Earnings Beat Model'
    )

    print(f"🎯 Model Confidence Metrics")
    print(f"   Brier Score: {confidence_result.brier_score:.4f} (lower is better, 0=perfect)")
    print(f"   Calibration Error (ECE): {confidence_result.calibration_error:.4f}")
    print(f"   Discrimination (AUC-ROC): {confidence_result.discrimination_auc:.3f}")
    print(f"   Overall Confidence: {confidence_result.overall_confidence:.1f}/100")


### 3.17 InferenceData Schema Integration

Build ArviZ-compatible InferenceData objects from all Bayesian category analyses
for standardised posterior diagnostics, R-hat convergence checks, and xarray/NetCDF export.


In [ ]:
# Build InferenceData from Bayesian category analysis results
if ARVIZ_AVAILABLE:
    print('🔬 Building InferenceData objects for all categories...')
    print()
    inference_data_objects = {}

    category_results_map = {
        'Valuation Ratios': ('val_results', 'Valuation Ratios'),
        'Profitability': ('prof_results', 'Profitability'),
        'Quality & Risk': ('quality_results', 'Quality & Risk'),
        'Leverage & Liquidity': ('leverage_results', 'Leverage & Liquidity'),
        'Analyst Sentiment': ('sentiment_results', 'Analyst Sentiment'),
        'Earnings Quality': ('earnings_quality_results', 'Earnings Quality'),
        'Growth Metrics': ('growth_results', 'Growth Metrics'),
        'Cash Flow': ('cashflow_results', 'Cash Flow'),
    }

    for label, (var_name, cat_name) in category_results_map.items():
        if var_name in dir() and eval(var_name):
            results = eval(var_name)
            features = [f for f in FeatureViewCatalog.get(cat_name, []) if f in results]
            if features:
                try:
                    idata = build_category_analysis_inference_data(results, df, cat_name, features)
                    summary = summarize_inference_data(idata)
                    inference_data_objects[label] = idata
                    rhat_info = ''
                    if summary.get('r_hat'):
                        max_rhat = max(summary['r_hat'].values())
                        rhat_info = f', max R-hat: {max_rhat:.4f}'
                    print(
                        f'  ✅ {label}: {summary.get("n_chains", 0)} chains × {summary.get("n_draws", 0)} draws{rhat_info}')
                except Exception as e:
                    print(f'  ⚠️ {label}: {e}')

    print(f'\n📊 Built {len(inference_data_objects)} InferenceData objects')
else:
    print('⚠️ ArviZ not available — skipping InferenceData builds')


## 4. Advanced Modeling: Hierarchical Bayes & Copulas
Modeling sector-level dependencies and tail correlations between Valuation and Quality.


In [ ]:
roe_hierarchical = hierarchical_mcmc_by_sector(df, 'roe', sector_col='industry')
copula_fit = fit_gaussian_copula(df, ['p_e_ratio', 'piotroski_f_score'])


## 5. Enhanced Visualizations
New advanced visualizations for comprehensive analysis including labor productivity and valuation ranges.


In [ ]:
# Valuation violin plot by industry
create_valuation_violin_plot(df).show()


In [ ]:
# Leverage vs liquidity bubble chart
create_leverage_liquidity_bubble_chart(df).show()


In [ ]:
# Productivity Quadrant
create_productivity_quadrant(df).show()


In [ ]:
# Valuation Range Visual for top stock
if len(df) > 0:
    top_ticker = df.iloc[0]['ticker']
    create_valuation_range_visual(df, top_ticker).show()


In [ ]:
# Quality radar chart for top stock
if len(df) > 0:
    top_ticker = df.iloc[0]['ticker']
    print(f"Quality Radar for: {top_ticker}")
    create_quality_risk_radar_chart(df, top_ticker).show()


### 5.1 Valuation Analysis Visualizations
Comprehensive valuation ratio analysis using the valuation module.


In [ ]:
# Valuation distribution dashboard - multi-panel violin plots
create_valuation_distribution_dashboard(df).show()


In [ ]:
# Relative valuation matrix - Z-score heatmap by industry
create_relative_valuation_matrix(df).show()


In [ ]:
# Valuation vs Growth quadrant - PEG-style analysis
create_valuation_vs_growth_quadrant(df).show()


In [ ]:
# Historical valuation percentile distribution
create_historical_valuation_percentile(df).show()


In [ ]:
# Valuation multiples comparison for top stock
if len(df) > 0:
    top_ticker = df.iloc[0]['ticker']
    print(f"Valuation Multiples for: {top_ticker}")
    create_valuation_multiples_comparison(df, ticker=top_ticker).show()


### 5.2 Earnings Quality Visualizations
Deep-dive earnings quality and predictability analysis.


In [ ]:
# Earnings surprise dashboard - multi-panel analysis
create_earnings_surprise_dashboard(df).show()


In [ ]:
# EPS trajectory analysis - improvement counts and streak analysis
create_eps_trajectory_analysis(df).show()


In [ ]:
# Beat rate heatmap by sector
create_beat_rate_heatmap(df).show()


In [ ]:
# Earnings consistency matrix
create_earnings_consistency_matrix(df).show()


In [ ]:
# Earnings quality decomposition for top stock
if len(df) > 0:
    top_ticker = df.iloc[0]['ticker']
    print(f"Earnings Quality Decomposition for: {top_ticker}")
    create_earnings_quality_decomposition(df, ticker=top_ticker).show()


### 5.3 Quality & Risk Visualizations
Comprehensive quality scoring and risk assessment.


In [ ]:
# Piotroski F-Score breakdown
create_piotroski_fscore_breakdown(df).show()


In [ ]:
# Altman Z-Score distribution with distress zones
create_altman_zscore_distribution(df).show()


In [ ]:
# Quality-Risk quadrant (F-Score vs Z-Score)
create_quality_risk_quadrant(df).show()


In [ ]:
# Beneish M-Score analysis with manipulation probability zones
create_beneish_mscore_analysis(df).show()


In [ ]:
# Risk tier sunburst (Sector → Industry → Risk Tier)
create_risk_tier_sunburst(df).show()


In [ ]:
# Distress early warning dashboard
create_distress_early_warning_dashboard(df).show()


### 5.4 Growth Analysis Visualizations
Comprehensive growth metrics analysis.


In [ ]:
# Growth consistency matrix by sector
create_growth_consistency_matrix(df).show()


In [ ]:
# Growth vs Profitability quadrant (BCG-style)
create_growth_vs_profitability_quadrant(df).show()


In [ ]:
# Growth acceleration chart
create_growth_acceleration_chart(df).show()


In [ ]:
# Sustainable growth analysis (SGR = ROE × Retention Rate)
create_sustainable_growth_analysis(df).show()


In [ ]:
# Growth waterfall chart for top stock
if len(df) > 0:
    top_ticker = df.iloc[0]['ticker']
    print(f"Growth Waterfall for: {top_ticker}")
    create_growth_waterfall_chart(df, ticker=top_ticker).show()


## 6. Stock Screening & Summary
Final ranking and interactive dashboard for the top opportunities using multiple screening strategies.


### 5.5 Probabilistic Visualizations

ArviZ-backed probabilistic charts for Bayesian category analysis and earnings beat posteriors.


In [ ]:
# Bayesian category ridge plot for Profitability
prof_features = [f for f in FeatureViewCatalog.get('Profitability', []) if f in df.columns][:5]
if prof_features:
    prof_results = bayesian_category_analysis(df, 'Profitability', prof_features)
    fig = create_bayesian_category_ridge(prof_results, category_name='Profitability')
    fig.show()


In [ ]:
# Beat probability posterior from Bayesian earnings model
if 'probability_results' in dir() and len(probability_results) > 0:
    fig = create_beat_probability_posterior(probability_results, top_n=1500)
    fig.show()
else:
    print('⚠️ probability_results not available — run section 3.16 first')


### 6.1 Enhanced Quality Screener


In [ ]:
# Quality screening with multiple criteria
quality_stocks = create_enhanced_screener(df, min_fscore=7, min_fcf_positive_years=4)
print(f"🏆 Quality Screen: {len(quality_stocks)} stocks found")
if len(quality_stocks) > 0:
    display(quality_stocks[['ticker', 'name', 'sector', 'industry', 'exchange', 'piotroski_f_score',
                            'distress_risk_score', 'fcf_positive_years']].head(50))


### 6.2 GARP (Growth at Reasonable Price) Screener


In [ ]:
# GARP opportunities
garp_stocks = screen_garp_opportunities(df)
print(f"📈 GARP Screen: {len(garp_stocks)} stocks found")
if len(garp_stocks) > 0:
    cols = ['ticker', 'name', 'sector', 'industry', 'country', 'exchange']
    if 'peg_ratio' in garp_stocks.columns:
        cols.append('peg_ratio')
    if 'eps_yoy_growth' in garp_stocks.columns:
        cols.append('eps_yoy_growth')
    if 'p_e_ratio' in garp_stocks.columns:
        cols.append('p_e_ratio')
    display(garp_stocks[cols].head(50))


### 6.3 High-Yield Safe Dividend Screener


In [ ]:
# Safe high-yield dividends
safe_div_stocks = screen_high_yield_safe_dividends(df)
print(f"💰 Safe High-Yield Dividend Screen: {len(safe_div_stocks)} stocks found")
if len(safe_div_stocks) > 0:
    yield_col = 'dividend_yield_ltm' if 'dividend_yield_ltm' in safe_div_stocks.columns else 'dividend_yield'
    cols = ['ticker', 'name', 'sector', 'industry', 'country', 'exchange']
    if 'dividend_payout_ratio' in safe_div_stocks.columns:
        cols.append('dividend_payout_ratio')
    if 'distress_risk_score' in safe_div_stocks.columns:
        cols.append('distress_risk_score')
    display(safe_div_stocks[cols].head(50))


### 6.4 Value Opportunities Screener


In [ ]:
# Value opportunities
value_stocks = screen_value_opportunities(df, max_pe_ratio=20, min_upside_potential=15)
print(f"💎 Value Screen: {len(value_stocks)} stocks found")
if len(value_stocks) > 0:
    cols = ['ticker', 'name', 'sector', 'industry', 'country', 'exchange']
    if 'p_e_ratio' in value_stocks.columns:
        cols.append('p_e_ratio')
    if 'upside_potential' in value_stocks.columns:
        cols.append('upside_potential')
    if 'fcf_yield' in value_stocks.columns:
        cols.append('fcf_yield')
    display(value_stocks[cols].head(50))


### 6.5 Growth Momentum Screener


In [ ]:
# Growth momentum stocks
growth_stocks = screen_growth_momentum(df, min_revenue_growth=5)
print(f"🚀 Growth Momentum Screen: {len(growth_stocks)} stocks found")
if len(growth_stocks) > 0:
    cols = ['ticker', 'name', 'industry']
    if 'revenue_yoy_growth' in growth_stocks.columns:
        cols.append('revenue_yoy_growth')
    if 'eps_yoy_growth' in growth_stocks.columns:
        cols.append('eps_yoy_growth')
    if 'long_term_trend_score' in growth_stocks.columns:
        cols.append('long_term_trend_score')
    display(growth_stocks[cols].head(15))


### 6.6 Valuation Reversion Screener


In [ ]:
# Valuation reversion opportunities
reversion_stocks = screen_valuation_reversion_candidates(df, min_discount_pct=25)
print(f"🔄 Valuation Reversion Screen: {len(reversion_stocks)} stocks found")
if len(reversion_stocks) > 0:
    cols = ['ticker', 'name', 'sector', 'industry', 'country', 'exchange']
    if 'p_e_vs_3y_avg' in reversion_stocks.columns:
        cols.append('p_e_vs_3y_avg')
    if 'ev_ebitda_vs_3y_avg' in reversion_stocks.columns:
        cols.append('ev_ebitda_vs_3y_avg')
    display(reversion_stocks[cols].head(50))


### 6.7 Integrity-Filtered Growth Screener


In [ ]:
# Integrity-filtered growth opportunities
integrity_growth_stocks = screen_integrity_filtered_growth(df, min_revenue_growth=5)
print(f"🛡️ Integrity-Filtered Growth Screen: {len(integrity_growth_stocks)} stocks found")
if len(integrity_growth_stocks) > 0:
    cols = ['ticker', 'name', 'sector', 'industry', 'country', 'exchange']
    if 'accounting_quality_score' in integrity_growth_stocks.columns:
        cols.append('accounting_quality_score')
    if 'dilution_score' in integrity_growth_stocks.columns:
        cols.append('dilution_score')
    if 'revenue_yoy_growth' in integrity_growth_stocks.columns:
        cols.append('revenue_yoy_growth')
    if 'eps_growth_yoy' in integrity_growth_stocks.columns:
        cols.append('eps_growth_yoy')
    display(integrity_growth_stocks[cols].head(50))


### 6.8 Summary Dashboard


In [ ]:
create_summary_dashboard(df).show()


### 6.9 Export Probability Analytics Results
Export earnings beat probability analysis, EPS streak analysis, and model confidence metrics to CSV files.


In [ ]:
output_dir = Path('outputs/analytics')
output_dir.mkdir(parents=True, exist_ok=True)

# Export probability analytics results
if len(probability_results) > 0 and len(streak_results) > 0:
    export_paths = export_probability_analytics_results(
        probability_df=probability_results,
        streak_df=streak_results,
        output_dir=output_dir,
        confidence_result=confidence_result if 'confidence_result' in dir() else None
    )
    print("📁 Exported Probability Analytics Results:")
    for name, path in export_paths.items():
        print(f"   ✓ {name}: {path}")


## 7. Export Results to Analytics Database
Persist probability analytics, screening results, and statistics to the `analytics` schema.


## 8. Analysis Summary


In [ ]:
# Optimization Summary
opt_status = get_optimization_status()
print("=" * 60)
print("📊 ANALYSIS COMPLETE")
print("=" * 60)
print(f"\n🔧 Environment:")
print(f"   JIT Acceleration: {opt_status.get('numba_available')}")
print(f"   Feature Categories: {len(FeatureViewCatalog)}")
print(f"   Total Stocks Analyzed: {len(df)}")

print(f"\n📈 Probability Analytics Summary:")
if len(probability_results) > 0:
    likely_beat = (probability_results['beat_classification'] == 'likely_beat').sum()
    print(f"   Earnings Beat Analysis: {len(probability_results)} stocks")
    print(f"   Likely Beat Classification: {likely_beat} stocks ({likely_beat / len(probability_results) * 100:.1f}%)")
    print(f"   Mean Posterior Beat Prob: {probability_results['posterior_beat_prob'].mean():.1%}")
    if 'dynamic_total_reports' in probability_results.columns:
        print(f"   Avg Dynamic Total Reports: {probability_results['dynamic_total_reports'].mean():.1f}")
if len(streak_results) > 0:
    print(f"   EPS Streak Analysis: {len(streak_results)} stocks")
    print(f"   Beat Streaks: {(streak_results['streak_type'] == 'beat').sum()}")
    print(f"   Miss Streaks: {(streak_results['streak_type'] == 'miss').sum()}")
    if 'dynamic_total_reports' in streak_results.columns:
        print(f"   Avg Dynamic Total Reports (Streak): {streak_results['dynamic_total_reports'].mean():.1f}")
if 'confidence_result' in dir():
    print(f"   Model Confidence Score: {confidence_result.overall_confidence:.1f}/100")

print(f"\n📁 Output Directory: outputs/analytics/")